# SurfaceTarget-CRC — phase 01 — patch 1 21

Extracted from the original Google Colab research notebook.

**Note:** Earlier patches and project data may be required to execute these cells.


In [ ]:
# ============================================================
# SurfaceTarget-CRC
# PHASE 1 — PATCH 1.21
#
# DIFFERENTIAL EXPRESSION ROBUSTNESS ASSESSMENT
#
# Sensitivity analysis:
# Repeat COAD Tier B vs normal differential expression
# after withholding previously QC-flagged samples.
#
# Objectives:
# 1. Load validated baseline DE results
# 2. Identify QC-flagged discovery samples
# 3. Create a QC-screened sensitivity cohort
# 4. Refit PyDESeq2 using original raw counts
# 5. Compare expression effect sizes
# 6. Assess significance and candidate stability
# 7. Preserve results and provenance
#
# IMPORTANT:
# No permanent sample exclusions.
# Original DE results remain unchanged.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
from zipfile import ZipFile, ZIP_DEFLATED

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

from scipy.stats import spearmanr
from packaging.version import Version

from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

META = ROOT / "data/metadata"
PROCESSED = ROOT / "data/processed"
RESULTS = ROOT / "results/tables"
FIGURES = ROOT / "results/figures"
CHECKPOINTS = ROOT / "checkpoints"

FIGURES.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
CHECKPOINTS.mkdir(parents=True, exist_ok=True)

MATRIX_PATH = (
    PROCESSED / "tcga_unstranded_raw_counts.npz"
)

BASELINE_PATH = (
    RESULTS /
    "tcga_coad_tierb_vs_normal_deseq2_results.csv"
)

DISCOVERY_PATH = (
    META / "tcga_coad_de_discovery_samples.csv"
)

QC_PATH = (
    RESULTS / "tcga_full_cohort_qc.csv"
)

print("=" * 65)
print("SURFACETARGET-CRC")
print("PATCH 1.21 — DIFFERENTIAL EXPRESSION ROBUSTNESS")
print("=" * 65)


# ------------------------------------------------------------
# 2. Validate software environment
# ------------------------------------------------------------

print("\nSoftware environment:")

print("PyDESeq2:", version("pydeseq2"))
print("Pandas:", pd.__version__)
print("AnnData:", version("anndata"))

if Version(pd.__version__) < Version("2.3.0"):

    raise RuntimeError(
        "Pandas >= 2.3.0 is required for "
        "the current AnnData environment."
    )

if version("pydeseq2") != "0.5.4":

    raise RuntimeError(
        "PyDESeq2 version differs from "
        "the baseline analysis."
    )


# ------------------------------------------------------------
# 3. Load existing research data
# ------------------------------------------------------------

for path in [
    MATRIX_PATH,
    BASELINE_PATH,
    DISCOVERY_PATH,
    QC_PATH
]:

    if not path.exists():

        raise FileNotFoundError(path)

with np.load(
    MATRIX_PATH,
    allow_pickle=False
) as archive:

    counts = archive["counts"]
    gene_ids = archive["gene_ids"]
    file_ids = archive["file_ids"]

baseline = pd.read_csv(
    BASELINE_PATH,
    dtype={"gene_id": str}
)

discovery = pd.read_csv(
    DISCOVERY_PATH,
    dtype=str,
    keep_default_na=False
)

qc = pd.read_csv(
    QC_PATH,
    dtype=str,
    keep_default_na=False
)

print("\nData loaded successfully.")

print("Expression matrix:", counts.shape)

print("Baseline genes:", len(baseline))

print("Discovery samples:", len(discovery))

print("QC samples:", len(qc))


# ------------------------------------------------------------
# 4. Validate baseline analysis
# ------------------------------------------------------------

if counts.shape != (60660, 472):

    raise RuntimeError(
        "Unexpected expression matrix dimensions."
    )

if len(baseline) != 14832:

    raise RuntimeError(
        "Unexpected baseline gene count."
    )

if len(discovery) != 223:

    raise RuntimeError(
        "Unexpected discovery sample count."
    )

if baseline["gene_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate gene IDs in baseline."
    )

if discovery["file_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate discovery sample IDs."
    )

if qc["file_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate QC file identifiers."
    )

if len(set(discovery["file_id"]) - set(file_ids)):

    raise RuntimeError(
        "Discovery samples missing from count matrix."
    )

if discovery["patient_id"].duplicated().any():

    raise RuntimeError(
        "Discovery cohort contains repeated patients."
    )

condition_counts = (
    discovery["condition"].value_counts()
)

if (
    condition_counts.get("Tumor", 0) != 194
    or condition_counts.get("Normal", 0) != 29
):

    raise RuntimeError(
        "Baseline cohort composition changed."
    )

print("\nBaseline validation: PASSED")


# ------------------------------------------------------------
# 5. Attach QC review flags
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("DISCOVERY COHORT QC AUDIT")
print("-" * 65)

qc_flags = qc[
    ["file_id", "n_qc_review_flags"]
].copy()

qc_flags["n_qc_review_flags"] = pd.to_numeric(
    qc_flags["n_qc_review_flags"],
    errors="raise"
).astype(int)

discovery = discovery.drop(
    columns=["n_qc_review_flags"],
    errors="ignore"
)

discovery = discovery.merge(
    qc_flags,
    on="file_id",
    how="left",
    validate="one_to_one"
)

if discovery["n_qc_review_flags"].isna().any():

    raise RuntimeError(
        "Missing QC flags for discovery samples."
    )

discovery["qc_flagged"] = (
    discovery["n_qc_review_flags"] > 0
)

discovery["included_in_sensitivity"] = (
    ~discovery["qc_flagged"]
)

print("\nOriginal discovery composition:")

print(
    discovery.groupby("condition")
    .agg(
        total=("file_id", "count"),
        flagged=("qc_flagged", "sum")
    )
    .to_string()
)

print(
    "\nTotal flagged discovery samples:",
    int(discovery["qc_flagged"].sum())
)

print("\nFlagged sample identifiers:")

flagged_samples = discovery[
    discovery["qc_flagged"]
].copy()

if not flagged_samples.empty:

    print(
        flagged_samples[
            [
                "patient_id",
                "sample_id",
                "condition",
                "n_qc_review_flags"
            ]
        ]
        .to_string(index=False)
    )

else:

    print("None.")


# ------------------------------------------------------------
# 6. Construct sensitivity cohort
# ------------------------------------------------------------

sensitivity_samples = discovery[
    discovery["included_in_sensitivity"]
].copy()

sensitivity_counts = (
    sensitivity_samples["condition"]
    .value_counts()
)

n_tumors = int(
    sensitivity_counts.get("Tumor", 0)
)

n_normals = int(
    sensitivity_counts.get("Normal", 0)
)

print("\n" + "-" * 65)
print("SENSITIVITY COHORT DEFINITION")
print("-" * 65)

print("\nRetained tumors:", n_tumors)

print("Retained normals:", n_normals)

print(
    "Total retained samples:",
    len(sensitivity_samples)
)

if n_tumors < 150 or n_normals < 20:

    raise RuntimeError(
        "Too few samples remain for the "
        "planned sensitivity comparison."
    )

if sensitivity_samples["patient_id"].duplicated().any():

    raise RuntimeError(
        "Repeated patient in sensitivity cohort."
    )

if sensitivity_samples["workflow_version"].nunique() != 1:

    raise RuntimeError(
        "Multiple workflow versions detected."
    )

print("\nSensitivity cohort validation: PASSED")


# ------------------------------------------------------------
# 7. Align matrix rows and columns
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING SENSITIVITY COUNT MATRIX")
print("-" * 65)

gene_index = pd.Index(gene_ids)

sample_index = pd.Index(file_ids)

gene_positions = gene_index.get_indexer(
    baseline["gene_id"].tolist()
)

sample_positions = sample_index.get_indexer(
    sensitivity_samples["file_id"].tolist()
)

if np.any(gene_positions < 0):

    raise RuntimeError(
        "Baseline genes missing from count matrix."
    )

if np.any(sample_positions < 0):

    raise RuntimeError(
        "Sensitivity samples missing from count matrix."
    )

# Extract original unmodified raw counts.
#
# Rows: genes
# Columns: retained samples

subset_counts = counts[
    np.ix_(
        gene_positions,
        sample_positions
    )
]

# Drop only genes with zero counts across
# all retained samples.
#
# This is necessary if a gene becomes entirely
# unexpressed after excluding QC-flagged samples.
#
# All remaining genes use the same baseline
# testing universe.

nonzero_gene_mask = (
    subset_counts.sum(axis=1) > 0
)

retained_gene_ids = (
    baseline["gene_id"]
    .to_numpy()[nonzero_gene_mask]
)

subset_counts = subset_counts[
    nonzero_gene_mask, :
]

print("\nBaseline gene universe:", len(baseline))

print(
    "Genes retained for sensitivity analysis:",
    len(retained_gene_ids)
)

print(
    "All-zero genes removed:",
    int((~nonzero_gene_mask).sum())
)

# PyDESeq2 expects samples x genes.

sensitivity_matrix = subset_counts.T

print(
    "\nSensitivity matrix:",
    sensitivity_matrix.shape
)

if np.any(sensitivity_matrix < 0):

    raise RuntimeError(
        "Negative raw counts detected."
    )


# ------------------------------------------------------------
# 8. Create PyDESeq2 input objects
# ------------------------------------------------------------

sample_ids = (
    sensitivity_samples["file_id"].tolist()
)

counts_df = pd.DataFrame(
    sensitivity_matrix,
    index=sample_ids,
    columns=retained_gene_ids
)

metadata = pd.DataFrame(
    {
        "condition":
            sensitivity_samples["condition"].tolist()
    },
    index=sample_ids
)

metadata["condition"] = pd.Categorical(
    metadata["condition"],
    categories=["Normal", "Tumor"],
    ordered=True
)

# Compatible string indexes for AnnData.

counts_df.index = pd.Index(
    counts_df.index,
    dtype="string"
)

counts_df.columns = pd.Index(
    counts_df.columns,
    dtype="string"
)

metadata.index = pd.Index(
    metadata.index,
    dtype="string"
)

if not counts_df.index.equals(metadata.index):

    raise RuntimeError(
        "Count and metadata indices are misaligned."
    )

print("\nPyDESeq2 input alignment: PASSED")


# ------------------------------------------------------------
# 9. Fit sensitivity differential expression model
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FITTING QC-SCREENED DIFFERENTIAL EXPRESSION")
print("=" * 65)

dds = DeseqDataSet(
    counts=counts_df,
    metadata=metadata,
    design="~condition",
    refit_cooks=True,
    n_cpus=2
)

dds.deseq2()

print("\nSensitivity model fitting: COMPLETE")


# ------------------------------------------------------------
# 10. Calculate sensitivity DE statistics
# ------------------------------------------------------------

stats = DeseqStats(
    dds,
    contrast=[
        "condition",
        "Tumor",
        "Normal"
    ],
    alpha=0.05,
    n_cpus=2
)

stats.summary()

sensitivity = (
    stats.results_df
    .copy()
    .rename_axis("gene_id")
    .reset_index()
)

if sensitivity["gene_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate genes in sensitivity results."
    )

print("\nSensitivity statistics: COMPLETE")


# ------------------------------------------------------------
# 11. Compare baseline and sensitivity results
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("BASELINE VS SENSITIVITY COMPARISON")
print("=" * 65)

baseline_comparison = baseline[
    [
        "gene_id",
        "gene_name",
        "log2FoldChange",
        "padj",
        "tier_a_median_cpm",
        "tier_a_fraction_cpm_ge1"
    ]
].copy()

baseline_comparison = baseline_comparison.rename(
    columns={
        "log2FoldChange":
            "baseline_log2fc",

        "padj":
            "baseline_padj"
    }
)

sensitivity_comparison = sensitivity[
    [
        "gene_id",
        "log2FoldChange",
        "padj"
    ]
].copy()

sensitivity_comparison = sensitivity_comparison.rename(
    columns={
        "log2FoldChange":
            "sensitivity_log2fc",

        "padj":
            "sensitivity_padj"
    }
)

comparison = baseline_comparison.merge(
    sensitivity_comparison,
    on="gene_id",
    how="left",
    validate="one_to_one"
)

comparison["baseline_upregulated"] = (
    comparison["baseline_padj"].lt(0.05)
    & comparison["baseline_log2fc"].gt(1)
)

comparison["sensitivity_upregulated"] = (
    comparison["sensitivity_padj"].lt(0.05)
    & comparison["sensitivity_log2fc"].gt(1)
)

comparison["tier_a_supported"] = (
    comparison["tier_a_fraction_cpm_ge1"].ge(0.50)
    & comparison["tier_a_median_cpm"].ge(1)
)

comparison["baseline_screening_candidate"] = (
    comparison["baseline_upregulated"]
    & comparison["tier_a_supported"]
)

comparison["robust_screening_candidate"] = (
    comparison["baseline_screening_candidate"]
    & comparison["sensitivity_upregulated"]
)

comparison["log2fc_difference"] = (
    comparison["sensitivity_log2fc"]
    - comparison["baseline_log2fc"]
)

# ------------------------------------------------------------
# 12. Calculate agreement statistics
# ------------------------------------------------------------

valid = comparison[
    comparison["baseline_log2fc"].notna()
    & comparison["sensitivity_log2fc"].notna()
].copy()

if len(valid) < 10000:

    raise RuntimeError(
        "Unexpectedly few comparable genes."
    )

rho, rho_pvalue = spearmanr(
    valid["baseline_log2fc"],
    valid["sensitivity_log2fc"]
)

baseline_up_count = int(
    comparison["baseline_upregulated"].sum()
)

sensitivity_up_count = int(
    comparison["sensitivity_upregulated"].sum()
)

baseline_candidate_count = int(
    comparison["baseline_screening_candidate"].sum()
)

robust_candidate_count = int(
    comparison["robust_screening_candidate"].sum()
)

stable_up_count = int(
    (
        comparison["baseline_upregulated"]
        & comparison["sensitivity_upregulated"]
    ).sum()
)

print("\nGenes compared:", len(valid))

print(
    "Spearman correlation of log2FC:",
    round(float(rho), 4)
)

print(
    "\nBaseline tumor-upregulated:",
    baseline_up_count
)

print(
    "Sensitivity tumor-upregulated:",
    sensitivity_up_count
)

print(
    "Baseline upregulated genes retaining "
    "significance and log2FC > 1:",
    stable_up_count
)

print(
    "\nOriginal Tier A-supported screening genes:",
    baseline_candidate_count
)

print(
    "Screening genes supported by both analyses:",
    robust_candidate_count
)


# ------------------------------------------------------------
# 13. Examine the original top 20 genes
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("TOP-GENE ROBUSTNESS REVIEW")
print("-" * 65)

top20 = comparison[
    comparison["baseline_screening_candidate"]
].sort_values(
    "baseline_padj"
).head(20).copy()

print("\nTop 20 original screening genes:")

print(
    top20[
        [
            "gene_name",
            "baseline_log2fc",
            "sensitivity_log2fc",
            "baseline_padj",
            "sensitivity_padj",
            "sensitivity_upregulated"
        ]
    ]
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.4g}"
    )
)

print(
    "\nTop 20 retaining sensitivity significance:",
    int(top20["sensitivity_upregulated"].sum())
)


# ------------------------------------------------------------
# 14. Plot baseline vs sensitivity effect sizes
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GENERATING ROBUSTNESS FIGURE")
print("-" * 65)

fig, ax = plt.subplots(
    figsize=(8.5, 8.5)
)

ax.scatter(
    valid["baseline_log2fc"],
    valid["sensitivity_log2fc"],
    s=9,
    alpha=0.22
)

limits = [
    min(
        valid["baseline_log2fc"].min(),
        valid["sensitivity_log2fc"].min()
    ),
    max(
        valid["baseline_log2fc"].max(),
        valid["sensitivity_log2fc"].max()
    )
]

ax.plot(
    limits,
    limits,
    linestyle="--",
    linewidth=1
)

ax.set_xlabel(
    "Baseline log2 fold change"
)

ax.set_ylabel(
    "QC-screened sensitivity log2 fold change"
)

ax.set_title(
    "TCGA-COAD — Differential Expression Robustness"
)

ax.grid(alpha=0.2)

fig.tight_layout()

figure_path = (
    FIGURES /
    "tcga_coad_qc_sensitivity_lfc_comparison.png"
)

fig.savefig(
    figure_path,
    dpi=200,
    bbox_inches="tight"
)

plt.show()

plt.close(fig)

print("\nRobustness figure saved.")


# ------------------------------------------------------------
# 15. Save analysis tables
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING SENSITIVITY RESULTS")
print("-" * 65)

sensitivity.to_csv(
    RESULTS /
    "tcga_coad_qc_sensitivity_deseq2_results.csv",
    index=False
)

comparison.to_csv(
    RESULTS /
    "tcga_coad_qc_sensitivity_comparison.csv",
    index=False
)

discovery.to_csv(
    META /
    "tcga_coad_qc_sensitivity_sample_manifest.csv",
    index=False
)

top20.to_csv(
    RESULTS /
    "tcga_coad_qc_sensitivity_top20_review.csv",
    index=False
)

size_factors = pd.DataFrame({
    "file_id": dds.obs.index.astype(str),
    "size_factor": dds.obs[
        "size_factors"
    ].to_numpy()
})

size_factors.to_csv(
    RESULTS /
    "tcga_coad_qc_sensitivity_size_factors.csv",
    index=False
)


# ------------------------------------------------------------
# 16. Save research provenance
# ------------------------------------------------------------

summary = {

    "project": "SurfaceTarget-CRC",

    "patch": "1.21",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "analysis": (
        "QC-screened sensitivity analysis "
        "of COAD Tier B vs normal"
    ),

    "software": "PyDESeq2",

    "software_version": version("pydeseq2"),

    "baseline_tumors": 194,

    "baseline_normals": 29,

    "sensitivity_tumors": n_tumors,

    "sensitivity_normals": n_normals,

    "qc_flagged_samples_withheld": int(
        discovery["qc_flagged"].sum()
    ),

    "baseline_genes": int(
        len(baseline)
    ),

    "sensitivity_genes": int(
        len(sensitivity)
    ),

    "baseline_upregulated_genes": baseline_up_count,

    "sensitivity_upregulated_genes": sensitivity_up_count,

    "stable_upregulated_genes": stable_up_count,

    "baseline_screening_candidates":
        baseline_candidate_count,

    "robust_screening_candidates":
        robust_candidate_count,

    "log2fc_spearman": float(rho),

    "model": "~condition",

    "contrast": [
        "condition",
        "Tumor",
        "Normal"
    ],

    "cook_outlier_refitting": True,

    "baseline_results_overwritten": False,

    "source_matrix_modified": False,

    "limitations": [
        "QC flags are provisional.",
        "Sample removal is a sensitivity analysis.",
        "No clinical covariates are adjusted.",
        "Tumor purity and tissue composition may confound results.",
        "Workflow matching does not eliminate every technical bias.",
        "Tier B is not independently verified MSS.",
        "Tier A support is descriptive expression prevalence.",
        "Statistical robustness does not establish targetability.",
        "Cook's-distance refitting remains enabled.",
        "Patient-matched sensitivity analysis remains pending."
    ]
}

with open(
    META /
    "tcga_coad_qc_sensitivity_provenance.json",
    "w"
) as output:

    json.dump(
        summary,
        output,
        indent=4
    )

print("\nSensitivity results saved.")


# ------------------------------------------------------------
# 17. Create research checkpoint
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CREATING RESEARCH CHECKPOINT")
print("-" * 65)

timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S")

checkpoint = (
    CHECKPOINTS /
    f"SurfaceTarget_CRC_PostDE_Patch121_{timestamp}.zip"
)

backup_folders = [
    "data/metadata",
    "data/processed",
    "results/tables",
    "results/figures",
    "src",
    "reports",
    "logs"
]

with ZipFile(
    checkpoint,
    "w",
    ZIP_DEFLATED,
    compresslevel=3
) as archive:

    for folder in backup_folders:

        folder_path = ROOT / folder

        if not folder_path.exists():
            continue

        for file_path in folder_path.rglob("*"):

            if file_path.is_file():

                archive.write(
                    file_path,
                    arcname=str(
                        file_path.relative_to(ROOT)
                    )
                )

checkpoint_mib = (
    checkpoint.stat().st_size
    / (1024 ** 2)
)

print("\nCheckpoint created.")

print(
    f"Checkpoint size: {checkpoint_mib:.2f} MiB"
)

print("Checkpoint location:", checkpoint)


# ------------------------------------------------------------
# 18. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 1.21 SUMMARY")
print("=" * 65)

print(
    "\nQC-flagged discovery samples withheld:",
    int(discovery["qc_flagged"].sum())
)

print("Retained tumors:", n_tumors)

print("Retained normals:", n_normals)

print("\nBaseline upregulated genes:", baseline_up_count)

print(
    "Sensitivity upregulated genes:",
    sensitivity_up_count
)

print(
    "Stable upregulated genes:",
    stable_up_count
)

print(
    "\nRobust Tier A-supported screening genes:",
    robust_candidate_count
)

print(
    "Log2FC Spearman correlation:",
    round(float(rho), 4)
)

print("\nSensitivity analysis: COMPLETE")

print("Research checkpoint: CREATED")

print("\nNo permanent sample exclusions.")

print("Original DE results preserved.")

print("\nPATCH 1.21 COMPLETE")

print("=" * 65)